# LifeMate Colab v3 — quy trình train có thể chạy lại

Notebook train agent Qwen3-4B từ nhãn **train**. ASR và đánh giá ToolEnv chạy ở bước khác.

**Mỗi lần mở runtime mới:** chỉnh duy nhất cell `Cấu hình`, sau đó chọn **Runtime → Run all**. Không chạy rời cell train.

```text
MyDrive/AI-ZALO/
├── dataset/
│   └── train+public_test/train/
│       ├── train.jsonl
│       └── train_tools_catalog.json
├── bundles/lifemate-colab-bundle-v3-*.tar.gz
└── lifemate_runs/<RUN_ID>/             # log, adapter, checkpoint, report
```

Notebook chỉ đọc `train/train.jsonl` và `train/train_tools_catalog.json` trong thư mục Drive; không dùng public/private test hoặc environment.

In [ ]:
# CẤU HÌNH — chỉ sửa các giá trị trong mục này
from pathlib import Path
import hashlib
import importlib
import json
import os
import shutil
import subprocess
import sys
import time

from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/AI-ZALO")
DATASET_ROOT = DRIVE_ROOT / "dataset/train+public_test"
DRIVE_TRAIN_DIR = DATASET_ROOT / "train"
DRIVE_TRAIN_JSONL = DRIVE_TRAIN_DIR / "train.jsonl"
DRIVE_TRAIN_CATALOG = DRIVE_TRAIN_DIR / "train_tools_catalog.json"
BUNDLE_DIR = DRIVE_ROOT / "bundles"
RUNS_DIR = DRIVE_ROOT / "lifemate_runs"
for folder in (DRIVE_ROOT / "dataset", BUNDLE_DIR, RUNS_DIR):
    folder.mkdir(parents=True, exist_ok=True)
BUNDLE_NAME = ""  # Trống = tự chọn bundle v3 mới nhất.

# prepare: dựng dữ liệu/preflight, không train
# smoke: chạy smoke train, adapter ghi lên Drive
# full: bắt đầu full train mới trong RUN_ID đã chọn
# resume: tiếp tục full train từ checkpoint đầy đủ gần nhất
ACTION = "prepare"
ATTEMPT_ID = time.strftime("%Y%m%d-%H%M%S")
RUN_ID = ""  # Trống = run mới; để chạy tiếp, nhập ID đã in ra.

# Siêu tham số: sửa tại đây cho các run mới.
SEED = 2026
LEARNING_RATE = 1e-4
FULL_EPOCHS = 2
GRADIENT_ACCUMULATION_STEPS = 16
SAVE_STEPS = 5
CONTEXT_CEILING = 8192
SMOKE_TRAIN_SAMPLES = 64
SMOKE_DEV_SAMPLES = 16

if ACTION not in {"prepare", "smoke", "full", "resume"}:
    raise ValueError("ACTION phải là prepare, smoke, full hoặc resume")
if not DRIVE_ROOT.is_dir():
    raise FileNotFoundError(f"Không thấy thư mục Drive chuẩn: {DRIVE_ROOT}")
missing_dataset_files = [p for p in (DRIVE_TRAIN_JSONL, DRIVE_TRAIN_CATALOG) if not p.is_file()]
if missing_dataset_files:
    paths = "\n".join(str(p) for p in missing_dataset_files)
    raise FileNotFoundError("Đưa hai file train vào đúng đường dẫn Drive:\n" + paths)
if BUNDLE_NAME:
    BUNDLE_PATH = BUNDLE_DIR / BUNDLE_NAME
else:
    bundles = sorted(BUNDLE_DIR.glob("lifemate-colab-bundle-v3-*.tar.gz"),
                     key=lambda p: p.stat().st_mtime, reverse=True)
    if not bundles:
        raise FileNotFoundError(f"Upload bundle v3 vào: {BUNDLE_DIR}")
    BUNDLE_PATH = bundles[0]
if not BUNDLE_PATH.is_file():
    raise FileNotFoundError(BUNDLE_PATH)
if ACTION in {"full", "resume"} and not RUN_ID:
    raise ValueError("ACTION=full/resume cần RUN_ID đã có smoke adapter")
if not RUN_ID:
    RUN_ID = time.strftime("%Y%m%d-%H%M%S")

RUN_DIR = RUNS_DIR / RUN_ID
if ACTION in {"full", "resume"} and not RUN_DIR.is_dir():
    raise FileNotFoundError(f"Run chưa tồn tại; kiểm tra RUN_ID: {RUN_DIR}")
RUN_DIR.mkdir(parents=True, exist_ok=True)

WORK_ROOT = Path("/content/lifemate_colab_v3")
PROJECT_DIR = WORK_ROOT / "lifemate"
TRAIN_DIR = PROJECT_DIR / "data/raw/train+public_test/train"
TRAIN_JSONL = PROJECT_DIR / "data/processed/sft/train.jsonl"
DEV_JSONL = PROJECT_DIR / "data/processed/sft/dev.jsonl"
MODEL_PATH = PROJECT_DIR / "models/Qwen3-4B-Instruct-2507"
SMOKE_OUTPUT_DIR = RUN_DIR / "lifemate-qwen3-4b-smoke-training"
LEGACY_SMOKE_ADAPTER = RUN_DIR / "lifemate-qwen3-4b-smoke-adapter"
SMOKE_ADAPTER = (LEGACY_SMOKE_ADAPTER if
                 (LEGACY_SMOKE_ADAPTER / "adapter_config.json").is_file()
                 else SMOKE_OUTPUT_DIR / "final")
FULL_OUTPUT_DIR = RUN_DIR / "lifemate-qwen3-4b-full-training"
FULL_ADAPTER = FULL_OUTPUT_DIR / "final"
MAX_SEQUENCE_LENGTH = None
GPU_INFO = None

print("Action:", ACTION)
print("Run ID:", RUN_ID)
print("Dataset train folder:", DATASET_ROOT)
print("Bundle:", BUNDLE_PATH.name)
print("Run output:", RUN_DIR)

## 1. Nạp source bundle

Cell này chạy lại an toàn sau khi runtime reset. Bundle chỉ chứa source/config, không chứa dataset hoặc weights.

In [ ]:
WORK_ROOT.mkdir(parents=True, exist_ok=True)
subprocess.run(["tar", "-xzf", str(BUNDLE_PATH), "-C", str(WORK_ROOT)], check=True)
if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(f"Bundle không có project tại {PROJECT_DIR}")
os.chdir(PROJECT_DIR)
subprocess.run(["sha256sum", "-c", "COLAB_BUNDLE_SHA256SUMS"],
               cwd=PROJECT_DIR, check=True)
sys.path.insert(0, str(PROJECT_DIR / "scripts"))
import colab_helpers
importlib.reload(colab_helpers)
run_logged = colab_helpers.run_logged
print("Source sẵn sàng:", PROJECT_DIR)

## 2. Đọc dataset từ Drive

Notebook chép đúng hai file train cần cho SFT từ Drive vào ổ làm việc Colab. Các file test, audio và environment không được mở.

In [ ]:
required_dataset_files = {
    "train.jsonl": DRIVE_TRAIN_JSONL,
    "train_tools_catalog.json": DRIVE_TRAIN_CATALOG,
}
missing = [str(path) for path in required_dataset_files.values() if not path.is_file()]
if missing:
    raise FileNotFoundError("Thiếu file dataset trên Drive:\n" + "\n".join(missing))

TRAIN_DIR.mkdir(parents=True, exist_ok=True)
for name, source_path in required_dataset_files.items():
    shutil.copy2(source_path, TRAIN_DIR / name)

sample_count = 0
with (TRAIN_DIR / "train.jsonl").open(encoding="utf-8") as stream:
    for line in stream:
        if line.strip():
            json.loads(line)
            sample_count += 1
with (TRAIN_DIR / "train_tools_catalog.json").open(encoding="utf-8") as stream:
    json.load(stream)
if sample_count == 0:
    raise ValueError("train.jsonl rỗng")

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

dataset_info = {
    "source_directory": str(DATASET_ROOT),
    "train_samples": sample_count,
    "train_jsonl_sha256": sha256_file(TRAIN_DIR / "train.jsonl"),
    "catalog_sha256": sha256_file(TRAIN_DIR / "train_tools_catalog.json"),
}
(RUN_DIR / "dataset_info.json").write_text(
    json.dumps(dataset_info, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
print(f"Dataset hợp lệ: {sample_count} train samples")
print("SHA256 train:", dataset_info["train_jsonl_sha256"])

## 3. GPU và thư viện

Giữ PyTorch/CUDA Colab cung cấp. Kiểm tra GPU, ổ đĩa, Trainer/PEFT, NF4 forward-backward và optimizer trước khi tải model lớn.

In [ ]:
subprocess.run(["nvidia-smi"], check=True)
free_gib = shutil.disk_usage("/content").free / 1024**3
if free_gib < 15:
    raise RuntimeError(f"Cần ít nhất 15 GiB ổ đĩa trống; hiện có {free_gib:.1f} GiB")

run_logged([sys.executable, "-u", "-m", "pip", "install", "-r", "requirements-train.txt"],
           PROJECT_DIR, RUN_DIR / f"install_{ATTEMPT_ID}.log")
run_logged([sys.executable, "-u", "scripts/colab_helpers.py"],
           PROJECT_DIR, RUN_DIR / f"runtime_check_{ATTEMPT_ID}.log")
info_code = (
    "import json, torch; p=torch.cuda.get_device_properties(0); "
    "print(json.dumps({'name':p.name,'vram_gib':p.total_memory/1024**3, "
    "'torch':torch.__version__,'cuda':torch.version.cuda, "
    "'native_bf16':torch.cuda.get_device_capability()[0]>=8 and torch.cuda.is_bf16_supported()}))"
)
GPU_INFO = json.loads(subprocess.check_output(
    [sys.executable, "-c", info_code], text=True
))
print("GPU:", GPU_INFO)
print(f"Ổ đĩa trống: {free_gib:.1f} GiB")

## 4. Tạo split và preflight

Tạo split cố định từ train, tải đúng revision Qwen và chọn context theo toàn bộ train/dev.

In [ ]:
run_logged([sys.executable, "-u", "scripts/download_models.py", "--models", "llm"],
           PROJECT_DIR, RUN_DIR / f"download_model_{ATTEMPT_ID}.log")
run_logged([
    sys.executable, "-u", "training/prepare_sft.py",
    "--data-root", str(PROJECT_DIR / "data/raw/train+public_test"),
    "--output-dir", str(PROJECT_DIR / "data/processed/sft"),
    "--seed", str(SEED),
], PROJECT_DIR, RUN_DIR / f"prepare_sft_{ATTEMPT_ID}.log")
manifest = json.loads((PROJECT_DIR / "data/processed/sft/manifest.json").read_text())
print("SFT samples:", manifest["samples"])
print("SFT next-action records:", manifest["records"])
(RUN_DIR / f"sft_manifest_{ATTEMPT_ID}.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)

run_logged([
    sys.executable, "-u", "training/train_lora.py",
    "--preflight-only", "--auto-context",
    "--max-sequence-length", str(CONTEXT_CEILING),
    "--model-path", str(MODEL_PATH),
    "--train-jsonl", str(TRAIN_JSONL), "--dev-jsonl", str(DEV_JSONL),
    "--preflight-report", str(RUN_DIR / f"preflight_full_{ATTEMPT_ID}.json"),
], PROJECT_DIR, RUN_DIR / f"preflight_full_{ATTEMPT_ID}.log")
preflight = json.loads((RUN_DIR / f"preflight_full_{ATTEMPT_ID}.json").read_text())
MAX_SEQUENCE_LENGTH = preflight["selected_context"]
print("Context theo dữ liệu:", MAX_SEQUENCE_LENGTH)
print("Số token tối đa:", {
    name: split["max_encoded_tokens"] for name, split in preflight["splits"].items()
})

## 5. Chạy action đã chọn

`prepare` chỉ chuẩn bị; `smoke` chạy mẫu nhỏ; `full` train mới; `resume` tiếp tục checkpoint. Full output/checkpoint nằm trên Drive. Smoke chỉ kiểm tra pipeline/GPU; full train khởi tạo từ Qwen gốc, không tiếp tục trọng số smoke. Đổi siêu tham số ở cell đầu; đổi `RUN_ID` trống để tạo thí nghiệm mới.

In [ ]:
def common_train_args(epochs):
    return [
        "--epochs", str(epochs),
        "--max-sequence-length", str(MAX_SEQUENCE_LENGTH),
        "--model-path", str(MODEL_PATH),
        "--train-jsonl", str(TRAIN_JSONL), "--dev-jsonl", str(DEV_JSONL),
        "--seed", str(SEED), "--learning-rate", str(LEARNING_RATE),
        "--gradient-accumulation-steps", str(GRADIENT_ACCUMULATION_STEPS),
        "--save-steps", str(SAVE_STEPS),
    ]

if ACTION == "prepare":
    print("Đã chuẩn bị xong. Chưa chạy train.")
elif ACTION == "smoke":
    if (SMOKE_ADAPTER / "adapter_config.json").is_file():
        raise FileExistsError(
            f"Run này đã có smoke adapter: {SMOKE_ADAPTER}. "
            "Chọn ACTION='full' hoặc dùng RUN_ID mới nếu muốn smoke lại."
        )
    command = [sys.executable, "-u", "training/train_lora.py"] + [
        "--limit-train-samples", str(SMOKE_TRAIN_SAMPLES),
        "--limit-dev-samples", str(SMOKE_DEV_SAMPLES), "--longest-first",
        "--output-dir", str(SMOKE_OUTPUT_DIR),
        "--preflight-report", str(RUN_DIR / f"preflight_smoke_{ATTEMPT_ID}.json"),
    ] + common_train_args(1)
    run_logged(command, PROJECT_DIR, RUN_DIR / f"smoke_train_{ATTEMPT_ID}.log")
    SMOKE_ADAPTER = SMOKE_OUTPUT_DIR / "final"
    if not (SMOKE_ADAPTER / "adapter_config.json").is_file():
        raise FileNotFoundError(f"Smoke xong nhưng thiếu adapter: {SMOKE_ADAPTER}")
    print("Smoke adapter đã nằm trên Drive:", SMOKE_ADAPTER)
elif ACTION in {"full", "resume"}:
    if not (SMOKE_ADAPTER / "adapter_config.json").is_file():
        raise FileNotFoundError(
            f"Không thấy smoke adapter trong run {RUN_ID}: {SMOKE_ADAPTER}. "
            "Đặt đúng RUN_ID hoặc chạy ACTION='smoke' trước."
        )
    command = [sys.executable, "-u", "training/train_lora.py"] + [
        "--output-dir", str(FULL_OUTPUT_DIR),
        "--preflight-report", str(RUN_DIR / f"preflight_full_training_{ATTEMPT_ID}.json"),
    ] + common_train_args(FULL_EPOCHS)
    if ACTION == "resume":
        checkpoint = colab_helpers.latest_complete_checkpoint(
            FULL_OUTPUT_DIR, require_scaler=not GPU_INFO["native_bf16"]
        )
        command += ["--resume-from-checkpoint", str(checkpoint)]
        log_path = RUN_DIR / f"full_resume_{ATTEMPT_ID}.log"
        print("Resume từ:", checkpoint)
    else:
        if FULL_OUTPUT_DIR.exists() and any(FULL_OUTPUT_DIR.iterdir()):
            raise FileExistsError(
                f"Full output đã có dữ liệu: {FULL_OUTPUT_DIR}. "
                "Nếu có checkpoint đầy đủ, chọn ACTION='resume'; "
                "nếu muốn train mới, dùng RUN_ID mới."
            )
        log_path = RUN_DIR / f"full_train_{ATTEMPT_ID}.log"
    run_logged(command, PROJECT_DIR, log_path)
    if not (FULL_ADAPTER / "adapter_config.json").is_file():
        raise FileNotFoundError(f"Full train xong nhưng thiếu adapter: {FULL_ADAPTER}")
    print("Full adapter đã lưu trên Drive:", FULL_ADAPTER)

run_info = {
    "run_id": RUN_ID, "action": ACTION, "gpu": GPU_INFO,
    "dataset": dataset_info, "bundle": BUNDLE_PATH.name,
    "max_sequence_length": MAX_SEQUENCE_LENGTH,
    "parameters": {
        "seed": SEED, "learning_rate": LEARNING_RATE,
        "full_epochs": FULL_EPOCHS,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "save_steps": SAVE_STEPS,
    },
    "smoke_adapter": str(SMOKE_ADAPTER) if (SMOKE_ADAPTER / "adapter_config.json").is_file() else None,
    "full_adapter": str(FULL_ADAPTER) if (FULL_ADAPTER / "adapter_config.json").is_file() else None,
}
(RUN_DIR / "run_info.json").write_text(
    json.dumps(run_info, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
print("Run artifacts:", RUN_DIR)

## Chạy lại sau khi ngắt Colab

Luôn mở lại notebook, đặt `ACTION` và `RUN_ID` ở cell đầu rồi chọn **Runtime → Run all**.

Để dùng smoke đã hoàn thành: `RUN_ID="20261105-114926"`, `ACTION="full"`. Nếu full train đã có checkpoint rồi bị ngắt: giữ run ID, đặt `ACTION="resume"`. Khi đổi learning rate/epoch/seed để làm thí nghiệm mới, để `RUN_ID=""` để tách output, không ghi đè adapter/checkpoint cũ.

Các log nằm trong thư mục run: `install_<ATTEMPT_ID>.log`, `runtime_check_<ATTEMPT_ID>.log`, `download_model_<ATTEMPT_ID>.log`, `prepare_sft_<ATTEMPT_ID>.log`, `preflight_*_<ATTEMPT_ID>.log`, `smoke_train_<ATTEMPT_ID>.log` hoặc `full_*_<ATTEMPT_ID>.log`. Dữ liệu và model trong `/content` được dựng/tải lại sau runtime reset; adapter, checkpoint, report và log trên Drive được giữ lại.